# Práctica de Laboratorio
AES y Modos de Operación sobre Imágenes BMP
Dra. Vanessa Miranda
Marzo 2026

## Objetivo
Comprender de forma visual y práctica cómo funcionan los modos de operación ECB, CBC y CTR al aplicarse sobre una imagen BMP, cómo se recupera la imagen mediante el proceso de descifrado con la llave correcta, qué sucede cuando se altera un byte del archivo cifrado y por qué la llave correcta es indispensable para recuperar la información.

## Competencias a desarrollar
- Identificar diferencias visuales entre ECB, CBC y CTR.
- Generar material criptográfico realista usando un generador aleatorio.
- Guardar y reutilizar llaves y parámetros criptográficos.
- Cifrar y descifrar imágenes BMP con AES.
- Analizar el efecto de modificar un byte del archivo cifrado.
- Verificar experimentalmente que solo la llave correcta permite recuperar la imagen.

## Material requerido
- Python 3 instalado.
- Una imagen en formato .bmp.
- Las librerías pycryptodome y pillow.


## Parte 0. Instalación de librerías
Ejecutar en terminal:
pip install pycryptodome pillow

Verificar:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import json

## Parte 1. Preparación del archivo de trabajo
Convertir la imagen a formato BMP si actualmente está en JPG o PNG.
El archivo de entrada deberá llamarse: input.bmp

TODO: colocar archivo input.bmp en el mismo directorio

## Parte 2. Código base

In [ ]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import json

In [ ]:
def pad(data, block_size=16):
    pad_len = block_size - (len(data) % block_size)
    return data + bytes([pad_len]) * pad_len

def read_bmp(file_path):
    with open(file_path, 'rb') as f:
        bmp = f.read()
    return bmp[:54], bmp[54:]

def write_bmp(file_path, header, body):
    with open(file_path, 'wb') as f:
        f.write(header + body)

### 2.2 Generación y guardado de material criptográfico

In [ ]:
def generate_crypto_material():
    return {
        'key': get_random_bytes(16).hex(),
        'iv': get_random_bytes(16).hex(),
        'nonce': get_random_bytes(8).hex()
    }

def save_crypto_material(material, file):
    with open(file, 'w') as f:
        json.dump(material, f, indent=4)

def load_crypto_material(file):
    with open(file, 'r') as f:
        return json.load(f)

### 2.3 Generar y guardar llaves
TODO: cambiar MATRICULA

In [ ]:
material = generate_crypto_material()
save_crypto_material(material, 'mis_claves_MATRICULA.json')
print(material)

## Parte 3. Cifrado de la imagen

In [ ]:
def encrypt_bmp(input_file, output_file, mode, material):
    header, body = read_bmp(input_file)
    key = bytes.fromhex(material['key'])

    if mode == 'ECB':
        cipher = AES.new(key, AES.MODE_ECB)
        enc = cipher.encrypt(pad(body))[:len(body)]
    elif mode == 'CBC':
        iv = bytes.fromhex(material['iv'])
        cipher = AES.new(key, AES.MODE_CBC, iv=iv)
        enc = cipher.encrypt(pad(body))[:len(body)]
    elif mode == 'CTR':
        nonce = bytes.fromhex(material['nonce'])
        cipher = AES.new(key, AES.MODE_CTR, nonce=nonce)
        enc = cipher.encrypt(body)
    else:
        raise ValueError('Modo no soportado')

    write_bmp(output_file, header, enc)

### 3.2 Cifrar la imagen en los tres modos
TODO: ejecutar

In [ ]:
material = load_crypto_material('mis_claves_MATRICULA.json')
encrypt_bmp('input.bmp','ecb.bmp','ECB',material)
encrypt_bmp('input.bmp','cbc.bmp','CBC',material)
encrypt_bmp('input.bmp','ctr.bmp','CTR',material)

### 3.3 Observación visual
Abrir los tres archivos generados:
ecb.bmp
cbc.bmp
ctr.bmp
y compáralos con la imagen original.

Preguntas de observación:
TODO: responder
1. Cuál de los tres modos deja ver más estructura de la imagen original?
2. Cuál parece producir una salida más cercana a ruido aleatorio?
3. Qué sugiere esto acerca de la seguridad de ECB frente a CBC y CTR?

## Parte 4 Descifrado

In [ ]:
def decrypt_bmp(input_file, output_file, mode, material):
    header, body = read_bmp(input_file)
    key = bytes.fromhex(material['key'])

    if mode == 'ECB':
        cipher = AES.new(key, AES.MODE_ECB)
        dec = cipher.decrypt(pad(body))[:len(body)]
    elif mode == 'CBC':
        iv = bytes.fromhex(material['iv'])
        cipher = AES.new(key, AES.MODE_CBC, iv=iv)
        dec = cipher.decrypt(pad(body))[:len(body)]
    elif mode == 'CTR':
        nonce = bytes.fromhex(material['nonce'])
        cipher = AES.new(key, AES.MODE_CTR, nonce=nonce)
        dec = cipher.decrypt(body)

    write_bmp(output_file, header, dec)

TODO: ejecutar descifrado y comparar visualmente

## Parte 4.2 Descifrar los archivos generados
TODO: ejecutar las siguientes líneas

In [ ]:
material = load_crypto_material('mis_claves_MATRICULA.json')
decrypt_bmp('ecb.bmp','ecb_descifrada.bmp','ECB',material)
decrypt_bmp('cbc.bmp','cbc_descifrada.bmp','CBC',material)
decrypt_bmp('ctr.bmp','ctr_descifrada.bmp','CTR',material)

### 4.3 Verificación
Abrir las imágenes descifradas y compáralas con la original.

Preguntas:
TODO: responder
4. Las imágenes descifradas coinciden visualmente con la original?
5. Por qué fue posible recuperar la imagen?

## Parte 5. Modificación de un byte del archivo cifrado

In [ ]:
def flip_byte(file_path, output_path, position):
    with open(file_path, 'rb') as f:
        data = bytearray(f.read())

    print('Byte original:', data[position])
    data[position] ^= 0xFF
    print('Byte modificado:', data[position])

    with open(output_path, 'wb') as f:
        f.write(data)

    print('Archivo modificado guardado en:', output_path)

### 5.2 Alterar una copia del archivo cifrado
Importante: no modificar los primeros 54 bytes, ya que corresponden al encabezado BMP.

TODO: ejecutar

In [ ]:
flip_byte('ecb.bmp','ecb_mod.bmp',2000)
flip_byte('cbc.bmp','cbc_mod.bmp',2000)
flip_byte('ctr.bmp','ctr_mod.bmp',2000)

## Parte 6. Descifrar después de modificar un byte
TODO: ejecutar

In [ ]:
material = load_crypto_material('mis_claves_MATRICULA.json')
decrypt_bmp('ecb_mod.bmp','ecb_mod_descifrada.bmp','ECB',material)
decrypt_bmp('cbc_mod.bmp','cbc_mod_descifrada.bmp','CBC',material)
decrypt_bmp('ctr_mod.bmp','ctr_mod_descifrada.bmp','CTR',material)

### Preguntas de análisis
TODO: responder
6. Qué efecto visual tuvo el cambio de un byte en ECB?
7. Qué diferencia observaste en CBC?
8. Qué diferencia observaste en CTR?
9. Qué modo propagó más el error?
10. Qué modo localizó mejor el error?

## Parte 7. Intercambio entre compañeros
En esta parte, se seleccionará al azar a otro compañero y se deberá compartir:
una imagen cifrada (por ejemplo, cbc.bmp)
su archivo mis_claves_MATRICULA.json

El compañero intentará descifrar la imagen con los materiales recibidos.

7.1 Descifrado con las llaves del compañero
TODO: ejecutar con archivos reales

7.2 Prueba adicional: intentar con la llave incorrecta
TODO: ejecutar prueba

## Parte 8. Conclusión individual
Redactar un párrafo final explicando, con tus propias palabras:

TODO: redactar
- cuál fue la diferencia principal entre ECB, CBC y CTR
- por qué ECB no es recomendable para información estructurada
- qué aprendiste sobre el uso correcto de la llave
- qué te mostró el experimento de modificación de un byte

## Entregables
1. Archivo mis_claves_MATRICULA.json
2. Imagen original input.bmp
3. Imágenes cifradas:
ecb.bmp
cbc.bmp
ctr.bmp
4. Imágenes descifradas:
ecb_descifrada.bmp
cbc_descifrada.bmp
ctr_descifrada.bmp
5. Imágenes alteradas y vueltas a descifrar:
ecb_mod.bmp, ecb_mod_descifrada.bmp
cbc_mod.bmp, cbc_mod_descifrada.bmp
ctr_mod.bmp, ctr_mod_descifrada.bmp
6. Respuestas a las preguntas de análisis
7. Conclusión individual

## Criterios de evaluación sugeridos
Criterio | Puntaje
--- | ---
Generación y guardado correcto de llaves y parámetros | 10 pts
Cifrado y descifrado correcto en ECB, CBC y CTR | 20 pts
Análisis visual de diferencias entre modos | 15 pts
Análisis del efecto de modificar un byte | 20 pts
Intercambio con compañero y uso correcto de llaves | 10 pts
Conclusión individual fundamentada | 25 pts
Total | 100 pts

## Nota final
En esta práctica, la llave no fue un valor teórico escrito manualmente, sino un dato generado de forma aleatoria. Cada estudiante tuvo su propia llave, y solo con esa llave fue posible recuperar correctamente la imagen cifrada. Además, al modificar un byte del archivo cifrado, se observó que cada modo de operación reacciona de forma distinta ante errores. Finalmente, al intercambiar archivos y llaves entre ustedes, se comprobó experimentalmente que el cifrado no solo transforma datos, sino que también controla quién puede recuperar la información.